# Exploratory data analysis for 2024 f1 race season

In [4]:
%run ../packages.py
import glob

##  Load Data for each race, combine into one dataframe 

In [23]:
# list of  races for 2024 season
races_2024 = ['bahrain',
              'saudi-arabian',
              'australian',
              'japanese',
              'chinese',
              'miami',
              'emilia-romagna',
              'monaco',
              'canadian',
              'spanish',
              'austrian',
              'british',
              'hungarian',
              'belgian',
              'dutch',
              'italian',
              'azerbaijan',
              'singapore',
              'united-states',
              'mexican',
              'sao-paulo',
              'las-vegas',
              'qatar',
              'abu-dhabi']

raw_path = '/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/raw'
race_file_data = glob.glob(f"{raw_path}/*.pkl")
existing_races_saved_2024 = [i.split('/')[-1].split('_')[1].split('.')[0]
                             for i in glob.glob(f"{raw_path}/*.pkl")]
# race_data = race_file_data[0]
# with open(race_data, "rb") as f:
#     race_data = pickle.load(f)

# # Check what you got
# print(f"Year: {race_data['year']}")
# print(f"Slug: {race_data['slug']}")
# print(f"URL: {race_data['url']}")

# # The list of DataFrames
# tables = race_data['tables']
# for i, df in enumerate(tables):
#     print(f"\nTable {i+1}:")
#     print(df.columns)

finishes = []
qualifying = []

for race_files in race_file_data:
    with open(race_files, "rb") as f:
        race_data = pickle.load(f)
    if race_files.split('/')[-1].split('_')[1].split('.')[0] in races_2024:
        print(f"Year: {race_data['year']}")
        print(f"Slug: {race_data['slug']}")
        print(f"URL: {race_data['url']}")
        tables = race_data['tables']
        finishes_df = tables[0]
        finishes_df['race_year'] = 2024
        finishes_df['race'] = race_data['slug']
        finishes.append(finishes_df)
        # qualifying data
        qualifying_df = tables[1]
        qualifying_df['race_year'] = 2024
        qualifying_df['race'] = race_data['slug']
        qualifying.append(qualifying_df)

finishes_df = pd.concat(finishes, ignore_index=True)
qualifying_df = pd.concat(qualifying, ignore_index=True)

Year: 2024
Slug: japanese
URL: https://pitwall.app/races/2024-japanese-grand-prix
Year: 2024
Slug: chinese
URL: https://pitwall.app/races/2024-chinese-grand-prix
Year: 2024
Slug: austrian
URL: https://pitwall.app/races/2024-austrian-grand-prix
Year: 2024
Slug: miami
URL: https://pitwall.app/races/2024-miami-grand-prix
Year: 2024
Slug: hungarian
URL: https://pitwall.app/races/2024-hungarian-grand-prix
Year: 2024
Slug: mexican
URL: https://pitwall.app/races/2024-mexican-grand-prix
Year: 2024
Slug: monaco
URL: https://pitwall.app/races/2024-monaco-grand-prix
Year: 2024
Slug: azerbaijan
URL: https://pitwall.app/races/2024-azerbaijan-grand-prix
Year: 2024
Slug: qatar
URL: https://pitwall.app/races/2024-qatar-grand-prix
Year: 2024
Slug: australian
URL: https://pitwall.app/races/2024-australian-grand-prix
Year: 2024
Slug: singapore
URL: https://pitwall.app/races/2024-singapore-grand-prix
Year: 2024
Slug: dutch
URL: https://pitwall.app/races/2024-dutch-grand-prix
Year: 2024
Slug: belgian
URL: 

In [61]:
from turtle import pos


def compute_prior(finishes, alpha=1):
    """Returns smoothed prior as a dict of position -> probability."""
    counts = pd.Series(finishes).value_counts().reindex(
        positions, fill_value=0)
    smoothed = (counts + alpha) / (len(finishes) + alpha * len(positions))
    return smoothed.to_dict()


# priors = {driver: compute_prior(last_season[driver]) for driver in drivers} /


# finishes_df['Pos.'].value_counts()
positions = list(range(1, 21))
positions = [str(pos) for pos in positions]
positions.extend(['DNF', 'DNS', 'DSQ'])
racer_finish_2024 = finishes_df.groupby('Driver')['Pos.'].apply(list).to_dict()
drivers = finishes_df['Driver'].unique().tolist()
priors = {driver: compute_prior(
    racer_finish_2024[driver]) for driver in drivers}

In [63]:
priors['#11Sergio Pérez']

{'1': 0.02127659574468085,
 '2': 0.0851063829787234,
 '3': 0.0425531914893617,
 '4': 0.0425531914893617,
 '5': 0.0425531914893617,
 '6': 0.0425531914893617,
 '7': 0.10638297872340426,
 '8': 0.0851063829787234,
 '9': 0.02127659574468085,
 '10': 0.06382978723404255,
 '11': 0.0425531914893617,
 '12': 0.02127659574468085,
 '13': 0.02127659574468085,
 '14': 0.02127659574468085,
 '15': 0.02127659574468085,
 '16': 0.02127659574468085,
 '17': 0.0851063829787234,
 '18': 0.02127659574468085,
 '19': 0.02127659574468085,
 '20': 0.02127659574468085,
 'DNF': 0.10638297872340426,
 'DNS': 0.02127659574468085,
 'DSQ': 0.02127659574468085}